# Optional Operational Extension: Day-2 Monitoring and AIOps

**Companion to** `multilingual_warranty_qwen3_vs_labse.ipynb`. The base notebook is the model-selection evaluation. This separate, optional notebook demonstrates an operating pattern a team can adopt after selecting one model. It is not part of the normal head-to-head decision path and does not represent a customer production implementation.

The extension captures embedding latency, evaluates cumulative quality, and scores each new synthetic batch against centroids learned only from earlier batches. That previous-batch baseline avoids leaking the current observations into their own drift score.

Flow: simulate weekly batches -> embed with latency capture -> previous-batch drift scoring -> quality gate -> optional billing KPIs -> MLflow evaluation history -> dashboard tables. There is no configured calendar schedule in this repository.

**Prerequisite:** run the evaluation notebook first with the same `catalog` and `schema` widget values. Attach to serverless compute. Optional billing data requires `system.billing.usage` access; the core path remains runnable without it. Synthetic data only.

## Contents
- 1. Dependencies and portable configuration
- 2. Batch simulation
- 3. Batch embedding and latency capture
- 4. Previous-batch drift monitor
- 5. Evaluation quality gate
- 6. Optional cost KPIs
- 7. MLflow lifecycle evidence
- 8. Dashboard outputs

## 1. Install deterministic evaluation dependencies

The monitoring notebook pins its clustering stack so interactive runs and bundle job runs use the same evaluation behavior.

In [ ]:
%pip install -q "scikit-learn==1.7.2" "threadpoolctl==3.6.0"
dbutils.library.restartPython()

## 1. Portable configuration

Bundle task parameters and interactive widgets use the same `catalog` and `schema` names. The code safely quotes both identifiers, verifies that the base table exists, and keeps all outputs in the selected namespace.

In [ ]:
dbutils.widgets.text("catalog", "", "Unity Catalog catalog (blank = current catalog)")
dbutils.widgets.text("schema", "multilingual_warranty_demo", "Target schema")

CATALOG = dbutils.widgets.get("catalog").strip() or spark.sql("SELECT current_catalog()").first()[0]
SCHEMA = dbutils.widgets.get("schema").strip()
if not CATALOG or not SCHEMA:
    raise ValueError("Both catalog and schema must resolve to non-empty names.")

def quote_identifier(value):
    if "\x00" in value:
        raise ValueError("Catalog and schema names cannot contain null bytes.")
    return f"`{value.replace('`', '``')}`"

def table_name(name):
    return f"{quote_identifier(CATALOG)}.{quote_identifier(SCHEMA)}.{quote_identifier(name)}"

NAMESPACE = f"{quote_identifier(CATALOG)}.{quote_identifier(SCHEMA)}"
EMB_MODEL = "databricks-qwen3-embedding-0-6b"
COST_TAG = "multilingual_warranty_embed"
MIN_CLUSTER_PURITY = 0.80
MIN_XLING_RETRIEVAL = 0.95
OOD_SIM_THRESHOLD = 0.45

spark.sql(f"USE CATALOG {quote_identifier(CATALOG)}")
spark.sql(f"USE SCHEMA {quote_identifier(SCHEMA)}")

BASE_TBL = table_name("warranty_claims")
if not spark.catalog.tableExists(BASE_TBL):
    raise RuntimeError(
        f"Missing {CATALOG}.{SCHEMA}.warranty_claims. Run the base notebook first "
        "with the same catalog and schema values."
    )
base = spark.table(BASE_TBL)
required = {"claim_id", "defect_id", "language", "claim_text"}
missing = required.difference(base.columns)
if missing:
    raise RuntimeError(f"Base table is missing required columns: {sorted(missing)}")
print("Base claims:", base.count())
display(base.limit(5))

## 2. Simulate deterministic weekly batches

The source corpus is divided into repeatable synthetic dates so the monitoring logic has a time axis without requiring external data.

In [ ]:
from pyspark.sql import Window, functions as F

# Deterministically spread each defect/language combination across six synthetic weeks.
# Production pipelines should use the source event or ingestion date instead.
N_BATCHES = 6
w = base.withColumn(
    "_rn",
    F.row_number().over(Window.partitionBy("defect_id", "language").orderBy("claim_id")) - 1,
)
batched = (
    w.withColumn(
        "batch_idx",
        F.pmod(F.xxhash64("claim_id", "defect_id", "language", "_rn"), F.lit(N_BATCHES)),
    )
    .withColumn("batch_date", F.date_add(F.lit("2026-07-06"), (F.col("batch_idx") * 7).cast("int")))
    .drop("_rn")
)

BATCHED_TBL = table_name("warranty_claims_batched")
batched.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(BATCHED_TBL)
display(
    spark.sql(
        f"SELECT batch_date, count(*) n_claims FROM {BATCHED_TBL} "
        "GROUP BY batch_date ORDER BY batch_date"
    )
)

## 3. Batch embedding and latency capture

Each simulated weekly batch is embedded with `ai_query`. The notebook records elapsed time and row count per batch, while the bundle applies a job tag that can be used for billing attribution when the runner has system-table access.

In [ ]:
import time

run_log = []
dates = [
    row.batch_date
    for row in spark.sql(
        f"SELECT DISTINCT batch_date FROM {BATCHED_TBL} ORDER BY batch_date"
    ).collect()
]
if not dates:
    raise RuntimeError("The batched source is empty; there is nothing to monitor.")

batch_embedding_tables = []
for batch_date in dates:
    output_table = table_name(f"_emb_batch_{batch_date.strftime('%Y%m%d')}")
    batch_embedding_tables.append(output_table)
    started = time.perf_counter()
    spark.sql(
        f"""
        CREATE OR REPLACE TABLE {output_table} AS
        SELECT claim_id, defect_id, language, batch_date,
               ai_query('{EMB_MODEL}', claim_text) AS embedding
        FROM {BATCHED_TBL}
        WHERE batch_date = DATE'{batch_date}'
        """
    )
    seconds = round(time.perf_counter() - started, 2)
    row_count = spark.table(output_table).count()
    run_log.append((batch_date, int(row_count), seconds, EMB_MODEL, COST_TAG))

EMB_RUN_LOG_TBL = table_name("embed_run_log")
emb_run_log = spark.createDataFrame(
    run_log,
    "batch_date date, n_rows int, seconds double, model string, cost_tag string",
)
emb_run_log.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(EMB_RUN_LOG_TBL)
display(
    spark.sql(
        f"SELECT *, round(seconds/greatest(n_rows, 1)*1000, 1) AS ms_per_row "
        f"FROM {EMB_RUN_LOG_TBL} ORDER BY batch_date"
    )
)

## 4. Previous-batch drift monitor

For each batch after the baseline week, the notebook fits centroids only on earlier observations and scores the new claims against that frozen baseline. A low best-centroid cosine is an emerging-issue candidate. The first batch establishes the baseline and is not scored as drift.

This is a demonstration threshold, not a calibrated production alert. Production teams should tune it on labeled historical incidents and monitor false-positive rates.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity

union_sql = " UNION ALL ".join(
    f"SELECT claim_id, defect_id, language, batch_date, embedding FROM {name}"
    for name in batch_embedding_tables
)
emb_df = spark.sql(union_sql).toPandas()
emb_df["embedding"] = emb_df["embedding"].apply(lambda value: np.asarray(value, dtype=float))
K = int(emb_df["defect_id"].nunique())

drift_rows = []
alert_rows = []
for batch_date in dates:
    current = emb_df[emb_df["batch_date"] == batch_date]
    prior = emb_df[emb_df["batch_date"] < batch_date]
    if prior.empty:
        drift_rows.append((batch_date, 0, int(len(current)), 0, None))
        continue

    prior_vectors = np.vstack(prior["embedding"].values)
    current_vectors = np.vstack(current["embedding"].values)
    n_clusters = min(K, len(prior_vectors))
    baseline = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(prior_vectors)
    best_similarity = cosine_similarity(current_vectors, baseline.cluster_centers_).max(axis=1)
    emerging = best_similarity < OOD_SIM_THRESHOLD

    for (_, row), score, is_emerging in zip(
        current.iterrows(), best_similarity, emerging
    ):
        if is_emerging:
            alert_rows.append(
                (
                    batch_date,
                    str(row["claim_id"]),
                    str(row["language"]),
                    float(round(score, 3)),
                )
            )
    drift_rows.append(
        (
            batch_date,
            int(n_clusters),
            int(len(current)),
            int(emerging.sum()),
            float(round(best_similarity.mean(), 3)),
        )
    )

DRIFT_TBL = table_name("drift_monitor")
ALERTS_TBL = table_name("emerging_issue_alerts")
spark.createDataFrame(
    drift_rows,
    "batch_date date, n_clusters int, n_new_claims int, n_emerging int, mean_best_similarity double",
).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(DRIFT_TBL)
spark.createDataFrame(
    alert_rows,
    "batch_date date, claim_id string, language string, best_similarity double",
).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(ALERTS_TBL)

display(spark.table(DRIFT_TBL).orderBy("batch_date"))

## 5. Evaluation quality gate

Each cumulative batch re-runs cluster purity, Adjusted Rand Index, and cross-lingual retrieval against the synthetic labels. The persisted `eval_history` supports trend monitoring, and the assertion fails the workflow when the latest batch falls below either configured gate.

In [ ]:
from sklearn.metrics import adjusted_rand_score

def cluster_purity(y_true, y_pred):
    label_index = {label: index for index, label in enumerate(sorted(set(y_true)))}
    true_index = np.array([label_index[label] for label in y_true])
    predicted = np.asarray(y_pred)
    matrix = np.zeros((len(label_index), predicted.max() + 1))
    for actual, cluster in zip(true_index, predicted):
        matrix[actual, cluster] += 1
    return matrix.max(axis=0).sum() / matrix.sum()

def xling_retrieval(vectors, y_true, languages):
    similarities = cosine_similarity(vectors)
    np.fill_diagonal(similarities, -1)
    hits = 0
    evaluated = 0
    for index in range(len(vectors)):
        candidates = np.where(np.asarray(languages) != languages[index])[0]
        if not len(candidates):
            continue
        neighbor = candidates[np.argmax(similarities[index, candidates])]
        hits += int(y_true[neighbor] == y_true[index])
        evaluated += 1
    return hits / max(evaluated, 1)

eval_rows = []
for batch_date in dates:
    cumulative = emb_df[emb_df["batch_date"] <= batch_date]
    vectors = np.vstack(cumulative["embedding"].values)
    labels = np.asarray(cumulative["defect_id"].tolist())
    languages = np.asarray(cumulative["language"].tolist())
    n_clusters = min(K, len(vectors))
    clusters = KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit(vectors)
    purity = round(cluster_purity(labels, clusters.labels_), 3)
    ari = round(adjusted_rand_score(labels, clusters.labels_), 3)
    retrieval = round(xling_retrieval(vectors, labels, languages), 3)
    passed = purity >= MIN_CLUSTER_PURITY and retrieval >= MIN_XLING_RETRIEVAL
    eval_rows.append((batch_date, purity, ari, retrieval, bool(passed)))

EVAL_HISTORY_TBL = table_name("eval_history")
eval_hist = spark.createDataFrame(
    eval_rows,
    "batch_date date, cluster_purity double, ari double, xling_retrieval double, passed boolean",
)
eval_hist.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(EVAL_HISTORY_TBL)
display(eval_hist.orderBy("batch_date"))

### Enforce the latest quality result

The workflow stops here when the newest cumulative batch misses either threshold, preventing later lifecycle logging from labeling a failed configuration as a candidate.

In [ ]:
latest = (
    spark.table(EVAL_HISTORY_TBL)
    .orderBy(F.col("batch_date").desc())
    .limit(1)
    .collect()[0]
)
assert latest["passed"], (
    f"QUALITY GATE FAILED for {latest['batch_date']}: "
    f"purity={latest['cluster_purity']} (minimum {MIN_CLUSTER_PURITY}), "
    f"retrieval={latest['xling_retrieval']} (minimum {MIN_XLING_RETRIEVAL})."
)
print("Quality gate PASSED for", latest["batch_date"])

## 6. Optional cost and latency KPIs

Latency always comes from `embed_run_log`. Billing usage is optional because many fork users cannot read `system.billing.usage`; when access or propagated job tags are unavailable, the notebook creates an empty typed table so the dashboard still deploys without inventing cost data.

In [ ]:
COST_BY_DAY_TBL = table_name("cost_by_day")
first_batch_date = min(dates).isoformat()
cost_sql = f"""
SELECT date(usage_start_time) AS usage_date,
       sum(usage_quantity) AS dbus
FROM system.billing.usage
WHERE custom_tags['pipeline'] = '{COST_TAG}'
  AND usage_start_time >= DATE'{first_batch_date}'
GROUP BY date(usage_start_time)
ORDER BY usage_date
"""
try:
    cost = spark.sql(cost_sql)
    cost.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(COST_BY_DAY_TBL)
    display(cost)
except Exception as error:
    print(
        "Billing usage unavailable; publishing an empty cost table. Reason:",
        str(error)[:200],
    )
    spark.sql(
        f"CREATE OR REPLACE TABLE {COST_BY_DAY_TBL} "
        "(usage_date date, dbus double)"
    )

display(
    spark.sql(
        f"SELECT batch_date, n_rows, seconds, "
        f"round(seconds/greatest(n_rows, 1)*1000, 1) AS ms_per_row "
        f"FROM {EMB_RUN_LOG_TBL} ORDER BY batch_date"
    )
)

## 7. MLflow lifecycle evidence

Qwen3 is a Databricks-managed endpoint, so registering an empty placeholder model in Unity Catalog would misrepresent what is being versioned. Instead, the notebook records the endpoint, thresholds, pinned evaluation inputs, and latest quality metrics in MLflow. A rollback means restoring a previously successful run configuration in the bundle, while endpoint model lifecycle remains provider-managed.

In [ ]:
import mlflow

configuration = {
    "embedding_model": EMB_MODEL,
    "catalog": CATALOG,
    "schema": SCHEMA,
    "n_clusters": int(K),
    "ood_similarity_threshold": OOD_SIM_THRESHOLD,
    "minimum_cluster_purity": MIN_CLUSTER_PURITY,
    "minimum_cross_lingual_retrieval": MIN_XLING_RETRIEVAL,
    "evaluation_batch_date": str(latest["batch_date"]),
}
metrics = {
    "cluster_purity": float(latest["cluster_purity"]),
    "cross_lingual_retrieval": float(latest["xling_retrieval"]),
    "adjusted_rand_index": float(latest["ari"]),
}

with mlflow.start_run(
    run_name=f"warranty-embedding-eval-{latest['batch_date']}",
    nested=mlflow.active_run() is not None,
) as lifecycle_run:
    mlflow.log_params(configuration)
    mlflow.log_metrics(metrics)
    mlflow.set_tags(
        {
            "pipeline": COST_TAG,
            "quality_gate": "passed",
            "lifecycle_status": "candidate",
        }
    )
    mlflow.log_dict(configuration, "embedding_configuration.json")
    print("Logged reproducible evaluation configuration:", lifecycle_run.info.run_id)

## 8. Dashboard outputs

Five stable tables feed the AI/BI dashboard. The bundle deploys the dashboard definition, and this job populates or refreshes its datasets.

In [ ]:
dashboard_tables = [
    "drift_monitor",
    "emerging_issue_alerts",
    "eval_history",
    "embed_run_log",
    "cost_by_day",
]
for table in dashboard_tables:
    qualified = table_name(table)
    row_count = spark.table(qualified).count()
    print(f"{CATALOG}.{SCHEMA}.{table:26s} {row_count:>6} rows")

## 9. Operational interpretation

- Drift scoring uses only prior batches as its baseline, preventing current-batch leakage.
- The quality gate fails the Databricks Workflow before downstream use when the latest cumulative metrics regress.
- Latency is always available; billing cost remains explicitly unavailable rather than fabricated when permissions or propagated tags are missing.
- MLflow records the evaluated endpoint configuration. The managed endpoint itself is not falsely presented as a user-registered UC model.

This synthetic demonstration is not a calibrated production detector. Thresholds, alert routing, retention, and rollback automation require validation on representative historical data.